# 04 - Backtest Analysis

Walk-forward backtest of the flagship **multi-asset rotation** strategy, a full **tearsheet**, the **Deflated Sharpe Ratio**, and **BHY multiple-testing** correction across strategy variants.

> **Data input:** Set exactly one of `QUANTCORTEX_PRICES_CSV` or `QUANTCORTEX_LIVE_YFINANCE=1`. No market data or executed outputs are committed.


In [ ]:
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("hmmlearn").setLevel(logging.ERROR)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the repository root on the path when running from research/.
for candidate in ("..", "."):
    absolute = os.path.abspath(candidate)
    if absolute not in sys.path:
        sys.path.insert(0, absolute)

from quantcortex.data.local_csv import load_ohlcv_csv, load_price_matrix

PRICE_CSV = os.environ.get("QUANTCORTEX_PRICES_CSV")
OHLCV_CSV = os.environ.get("QUANTCORTEX_OHLCV_CSV")
LIVE_YFINANCE = os.environ.get("QUANTCORTEX_LIVE_YFINANCE") == "1"

if (PRICE_CSV is not None) == LIVE_YFINANCE:
    raise RuntimeError(
        "Set exactly one of QUANTCORTEX_PRICES_CSV=/path/to/prices.csv "
        "or QUANTCORTEX_LIVE_YFINANCE=1."
    )


def load_prices(symbols, start="2018-01-01"):
    """Load adjusted closes from the explicitly selected real-data source."""
    if PRICE_CSV is not None:
        return load_price_matrix(PRICE_CSV, symbols=list(symbols), start=start)

    print(
        "Using live Yahoo Finance data through yfinance; review the provider "
        "terms and confirm that your use is permitted."
    )
    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    prices = YFinanceProvider().get_prices(list(symbols), start=start)
    if prices is None or prices.empty:
        raise RuntimeError("yfinance returned no prices")
    prices = prices.dropna(how="all").ffill().dropna()
    if prices.shape[0] <= 120:
        raise RuntimeError("insufficient complete price history from yfinance")
    return prices


def load_ohlcv(symbol, start="2018-01-01"):
    """Load one symbol's real OHLCV data for feature engineering."""
    if OHLCV_CSV is not None:
        return load_ohlcv_csv(OHLCV_CSV, start=start)
    if not LIVE_YFINANCE:
        raise RuntimeError(
            "Set QUANTCORTEX_OHLCV_CSV=/path/to/ohlcv.csv for the Alpha158 cell."
        )

    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    frame = YFinanceProvider().fetch_ohlcv([symbol], start=start).get(symbol)
    if frame is None or frame.empty:
        raise RuntimeError(f"yfinance returned no OHLCV data for {symbol}")
    return frame.dropna()


selected_source = (
    f"local CSV {Path(PRICE_CSV).expanduser().resolve()}"
    if PRICE_CSV is not None
    else "explicit live yfinance download"
)
print(f"quantcortex research environment ready; source: {selected_source}")


In [ ]:
symbols = ["QQQ","VGT","GLD","TLT","SPY","VIG"]
prices = load_prices(symbols)
print("rotation universe:", list(prices.columns), prices.shape)

## Generate weekly target weights

In [ ]:
from quantcortex.strategies.multi_asset_rotation import MultiAssetRotation

weekly = prices.index[prices.index.weekday == 0]      # Mondays
strat = MultiAssetRotation()
W = strat.generate_weights(prices, weekly)
print("weekly target-weight panel:", W.shape)
assert bool((W.sum(axis=1) <= 1.0 + 1e-6).all())
W.tail(4).round(3)

## Run the backtest with transaction costs

In [ ]:
from quantcortex.backtest.costs.transaction_costs import TransactionCostModel
from quantcortex.backtest.engines.vectorized import VectorizedBacktest

result = VectorizedBacktest(TransactionCostModel()).run(W, prices)
rets = result.returns.dropna()
print(result.summary())

## Tearsheet

In [ ]:
from quantcortex.backtest.metrics.tearsheet import Tearsheet

ts = Tearsheet(rets)
metrics = ts.compute()
for k in ["cagr","ann_vol","sharpe","sortino","calmar","max_drawdown","var_95","cvar_95"]:
    print(f"{k:>14}: {metrics[k]:+.4f}")

## Deflated Sharpe Ratio (overfitting control)

In [ ]:
from quantcortex.backtest.validation.deflated_sharpe import compute_dsr, probabilistic_sharpe_ratio

for n in [1, 10, 50, 200]:
    print(f"DSR (n_trials={n:>3}): {compute_dsr(rets, n_trials=n):.4f}")
print(f"PSR            : {probabilistic_sharpe_ratio(rets):.4f}")

## BHY multiple-testing correction across variants

In [ ]:
from scipy import stats

from quantcortex.backtest.validation.multiple_testing import MultipleTestingReport, bhy_correction

variants = {"mom_63": 63, "mom_126": 126, "mom_252": 252}
pvals, labels = [], []
for label, lb in variants.items():
    s = MultiAssetRotation(mom_lookback=lb)
    w = s.generate_weights(prices, weekly)
    r = VectorizedBacktest(TransactionCostModel()).run(w, prices).returns.dropna()
    t = r.mean() / (r.std(ddof=1) + 1e-12) * np.sqrt(len(r))
    pvals.append(float(2 * (1 - stats.norm.cdf(abs(t))))); labels.append(label)
reject, adj = bhy_correction(pvals, alpha=0.05)
print(MultipleTestingReport(pvals, labels=labels).summary())

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
result.equity_curve.plot(ax=ax[0]); ax[0].set_title("Strategy equity"); ax[0].set_ylabel("NAV")
dd = ts.drawdown_series()
dd.plot(ax=ax[1], color="crimson"); ax[1].fill_between(dd.index, dd.values, 0, color="crimson", alpha=0.3)
ax[1].set_title("Drawdown"); plt.tight_layout(); print("backtest analysis complete.")